# Malayalam Phase 1 — Step 5: Generation (Qwen2.5-7B-Instruct)

Mirrors English `05_generation.ipynb` with key optimizations for Malayalam:
1. **Model**: `Qwen/Qwen2.5-7B-Instruct` for strong multilingual reasoning and extraction.
2. **Script-Aware Normalizer**: Unicode NFC, Chillu standardization, ZWNJ/ZWJ removal, punctuation cleaning.
3. **Context Window**: Top-3 passages with 2,500 character limit (prevents cutting off gold answers located in the middle/end of Malayalam Wikipedia articles).
4. **Prompt**: Concise Malayalam instruction without refusal collapse traps.
5. **Evaluation**: Evaluates EM, F1, Contains-match, and Gold-passage ceiling.

## 1. GitHub setup

In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
print("Token loaded:", token is not None)

In [ ]:
import os

os.environ["GITHUB_TOKEN"] = token
!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git

In [ ]:
%cd /content/multilingual-rag-research

In [ ]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")

In [ ]:
!git fetch origin
!git switch nehna
!git pull

## 2. Load data

In [ ]:
import json

with open("data/questions_ml.json", "r", encoding="utf-8") as f:
    questions_ml = json.load(f)

with open("data/corpus_ml.json", "r", encoding="utf-8") as f:
    corpus_ml = json.load(f)

with open("results/hybrid_top10_ml.json", "r", encoding="utf-8") as f:
    hybrid_results_ml = json.load(f)

print("Questions   :", len(questions_ml))
print("Corpus      :", len(corpus_ml))
print("Hybrid IDs  :", len(hybrid_results_ml))

## 3. Install dependencies & load model

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Qwen2.5-7B-Instruct
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model: float16 if enough VRAM; fallback to 4-bit quantization on Colab T4
try:
    llm = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    print("Model loaded in float16:", model_name)
except Exception as e:
    print("Float16 loading failed or exceeded VRAM, loading in 4-bit:", e)
    llm = AutoModelForCausalLM.from_pretrained(
        model_name,
        load_in_4bit=True,
        device_map="auto"
    )
    print("Model loaded in 4-bit:", model_name)

## 4. Malayalam prompt & generation function

- **Passage budget**: Top-3 passages, each up to 2,500 characters (preserving sentence boundaries).
- **Prompt**: Direct Malayalam instruction focusing on extractive answering.

In [ ]:
MAX_PASSAGE_CHARS = 2500

def truncate_passage(text, max_chars=MAX_PASSAGE_CHARS):
    """Truncate passage at sentence boundary to preserve answer integrity."""
    if len(text) <= max_chars:
        return text
    # Cut at last full stop before limit
    cut = text[:max_chars].rfind(".")
    if cut > max_chars // 2:
        return text[:cut + 1]
    return text[:max_chars]

def build_prompt_ml(question, passages):
    """Construct Malayalam prompt with top-3 passages (up to 2500 chars each)."""
    truncated = [truncate_passage(p, max_chars=MAX_PASSAGE_CHARS) for p in passages[:3]]
    context = "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(truncated))

    return f"""താഴെ നൽകിയിരിക്കുന്ന വിവരങ്ങൾ അടിസ്ഥാനമാക്കി ചോദ്യത്തിന് ഉത്തരം നൽകുക.
ഉത്തരം ഒന്നോ രണ്ടോ വാക്കുകളിൽ വളരെ ചുരുക്കി എഴുതുക. അധിക വിശദീകരണങ്ങൾ ഒഴിവാക്കുക.

വിവരങ്ങൾ:
{context}

ചോദ്യം: {question}

ഉത്തരം:"""

def generate_answer_ml(question, passages, max_new_tokens=40):
    prompt = build_prompt_ml(question, passages)
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## 5. Sanity check on 3 examples

In [ ]:
for q in questions_ml[:3]:
    qid = q["question_id"]
    top_ids = hybrid_results_ml[str(qid)][:3]
    passages = [corpus_ml[pid]["text"] for pid in top_ids]

    ans = generate_answer_ml(q["question"], passages)
    print("=" * 70)
    print("Question :", q["question"])
    print("Gold     :", q["gold_answers"])
    print("Answer   :", ans)
print("=" * 70)

## 6. Run generation on all 100 questions (hybrid retrieval)

In [ ]:
predictions_ml = {}

for i, q in enumerate(questions_ml):
    qid = q["question_id"]
    top_ids = hybrid_results_ml[str(qid)][:3]
    passages = [corpus_ml[pid]["text"] for pid in top_ids]

    predictions_ml[qid] = generate_answer_ml(q["question"], passages)

    if i % 10 == 0:
        print(f"{i}/100 done")

print("Done. Total predictions:", len(predictions_ml))

## 7. Script-Aware Malayalam Normalization & Evaluation Metrics

In [ ]:
import re
import unicodedata

# Chillu mapping: legacy base+virama+ZWJ sequences to modern atomic chillu characters
CHILLU_MAP = {
    '\u0D23\u0D4D\u200D': '\u0D7A',  # ൺ
    '\u0D28\u0D4D\u200D': '\u0D7B',  # ൻ
    '\u0D30\u0D4D\u200D': '\u0D7C',  # ർ
    '\u0D32\u0D4D\u200D': '\u0D7D',  # ൽ
    '\u0D33\u0D4D\u200D': '\u0D7E',  # ൾ
    '\u0D15\u0D4D\u200D': '\u0D7F',  # ൿ
}

def normalize_answer_ml(text):
    """Normalize Malayalam answer:
    1. Unicode NFC normalization (first line)
    2. Standardize chillu characters
    3. Strip Zero Width Non-Joiner (ZWNJ \u200c) and Zero Width Joiner (ZWJ \u200d)
    4. Strip punctuation, hyphens, brackets
    5. Collapse whitespace
    """
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    for legacy, atomic in CHILLU_MAP.items():
        text = text.replace(legacy, atomic)
    text = re.sub(r"[\u200c\u200d]", "", text)
    text = re.sub(r"[?!\.,;:।\-\"\'\(\)\[\]]", " ", text)
    text = " ".join(text.split())
    return text

def exact_match_ml(prediction, gold_answers):
    pred_norm = normalize_answer_ml(prediction)
    for gold in gold_answers:
        if pred_norm == normalize_answer_ml(gold):
            return 1
    return 0

def f1_score_ml(prediction, gold):
    pred_tokens = normalize_answer_ml(prediction).split()
    gold_tokens = normalize_answer_ml(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)

    common = {}
    for t in pred_tokens:
        common[t] = common.get(t, 0) + 1

    gold_counts = {}
    for t in gold_tokens:
        gold_counts[t] = gold_counts.get(t, 0) + 1

    overlap = sum(
        min(count, common.get(token, 0))
        for token, count in gold_counts.items()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

def best_f1_ml(prediction, gold_answers):
    return max(f1_score_ml(prediction, gold) for gold in gold_answers)

def contains_match_ml(prediction, gold_answers):
    pred_norm = normalize_answer_ml(prediction)
    for gold in gold_answers:
        if normalize_answer_ml(gold) in pred_norm:
            return 1
    return 0

In [ ]:
em_scores_ml = []
f1_scores_ml = []
contains_scores_ml = []

for q in questions_ml:
    pred = predictions_ml[q["question_id"]]
    em_scores_ml.append(exact_match_ml(pred, q["gold_answers"]))
    f1_scores_ml.append(best_f1_ml(pred, q["gold_answers"]))
    contains_scores_ml.append(contains_match_ml(pred, q["gold_answers"]))

em_hybrid = sum(em_scores_ml) / len(em_scores_ml)
f1_hybrid = sum(f1_scores_ml) / len(f1_scores_ml)
contains_hybrid = sum(contains_scores_ml) / len(contains_scores_ml)

print("--- Hybrid Retrieval ---")
print(f"Exact Match    : {em_hybrid:.4f}")
print(f"F1             : {f1_hybrid:.4f}")
print(f"Contains-match : {contains_hybrid:.4f}")

## 8. Gold-passage ceiling experiment

In [ ]:
gold_predictions_ml = {}

for i, q in enumerate(questions_ml):
    gold_passage = corpus_ml[q["gold_passage_id"]]["text"]
    gold_predictions_ml[q["question_id"]] = generate_answer_ml(
        q["question"], [gold_passage]
    )

    if i % 10 == 0:
        print(f"{i}/100 done")

print("Done. Total gold predictions:", len(gold_predictions_ml))

In [ ]:
gold_em_ml = [
    exact_match_ml(gold_predictions_ml[q["question_id"]], q["gold_answers"])
    for q in questions_ml
]
gold_f1_ml = [
    best_f1_ml(gold_predictions_ml[q["question_id"]], q["gold_answers"])
    for q in questions_ml
]

em_gold = sum(gold_em_ml) / len(gold_em_ml)
f1_gold = sum(gold_f1_ml) / len(gold_f1_ml)

print("--- Gold Passage Ceiling ---")
print(f"Exact Match : {em_gold:.4f}")
print(f"F1          : {f1_gold:.4f}")

print("\n--- Retrieval Gap ---")
print(f"EM gap  : {(em_gold - em_hybrid)*100:.1f} pp")
print(f"F1 gap  : {(f1_gold - f1_hybrid)*100:.1f} pp")

In [ ]:
refusals_ml = sum(
    1 for q in questions_ml
    if "എനിക്കറിയില്ല" in str(predictions_ml[q["question_id"]])
    or "i don't know" in str(predictions_ml[q["question_id"]]).lower()
)
print(f"Refusals: {refusals_ml} out of {len(questions_ml)}")

## 9. Save results

In [ ]:
import os
import csv

os.makedirs("results", exist_ok=True)

with open("results/predictions_hybrid_ml.json", "w", encoding="utf-8") as f:
    json.dump(predictions_ml, f, ensure_ascii=False, indent=2)

with open("results/predictions_gold_ml.json", "w", encoding="utf-8") as f:
    json.dump(gold_predictions_ml, f, ensure_ascii=False, indent=2)

with open("results/generation_experiments_ml.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["run_name", "retrieval_method", "model_name", "em", "f1", "contains_match", "refusals"])
    writer.writerow([
        "hybrid_generation_ml",
        "hybrid",
        model_name,
        round(em_hybrid, 6),
        round(f1_hybrid, 6),
        round(contains_hybrid, 6),
        refusals_ml
    ])

print("Saved: predictions_hybrid_ml.json")
print("Saved: predictions_gold_ml.json")
print("Saved: generation_experiments_ml.csv")

## 10. Push to GitHub

In [ ]:
!git config --global user.email "nehnamehranmk17@gmail.com"
!git config --global user.name "nehnamehrankmk638-dev"

!git add notebooks/ml_05_generation.ipynb \
          results/predictions_hybrid_ml.json \
          results/predictions_gold_ml.json \
          results/generation_experiments_ml.csv

!git commit -m "ml phase1: updated generation notebook with Qwen2.5-7B, 2500-char context, and Malayalam extraction prompt"
!git push